setup

In [ ]:
import time, zlib
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen3-4B"
TAG = MODEL.split("/")[-1].lower()          # "qwen3-4b": used in cache filenames, so models never share a file
RUN_SWEEP = False                           # the 18-generation quality sweep; False = only load it if already saved
device = "mps" if torch.backends.mps.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to(device)
model.eval()

pilot = pd.read_csv("../data/prompts/b1_pilot.csv")

THINK_END = tok.convert_tokens_to_ids("</think>")
EOS_IDS = set(model.generation_config.eos_token_id)
SETTINGS = {
    "mclm_math100": dict(thinking=True,  max_new_tokens=4096, temperature=0.6, top_p=0.95, top_k=20),
    "lcb_dolly":    dict(thinking=False, max_new_tokens=256,  temperature=0.7, top_p=0.8,  top_k=20),
}
print(device, model.dtype, THINK_END, EOS_IDS)

generate one row and split it (row[...] access: `row.item` is the Series.item() method, not the column)

In [ ]:
def seed_for(source, item, lang, sample):
    return zlib.crc32(f"{source}|{item}|{lang}|{sample}".encode())

def split_think(gen_ids, thinking):
    """gen_ids: list[int] of NEW tokens only. Returns (trace_ids, answer_ids, truncated)."""
    ids = [t for t in gen_ids if t not in EOS_IDS]
    if not thinking:
        return [], ids, False
    if THINK_END not in ids:
        return ids, [], True
    k = ids.index(THINK_END)
    return ids[:k], ids[k + 1:], False

def distinct_ngram_ratio(ids, n=4):
    """Unique n-grams / total n-grams. ~1 = varied text, low (<0.3) = looping."""
    grams = [tuple(ids[i:i + n]) for i in range(len(ids) - n + 1)]
    return round(len(set(grams)) / len(grams), 2) if grams else None

@torch.no_grad()
def generate_one(row, sample=0):
    cfg = SETTINGS[row["source"]]
    msgs = [{"role": "user", "content": row["prompt"]}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  enable_thinking=cfg["thinking"],
                                  return_tensors="pt", return_dict=True).to(device)
    prompt_len = enc["input_ids"].shape[1]

    torch.manual_seed(seed_for(row["source"], row["item"], row["lang"], sample))
    t0 = time.perf_counter()
    out = model.generate(**enc, do_sample=True,
                         max_new_tokens=cfg["max_new_tokens"],
                         temperature=cfg["temperature"],
                         top_p=cfg["top_p"], top_k=cfg["top_k"])
    if device == "mps":
        torch.mps.synchronize()
    secs = time.perf_counter() - t0

    gen_ids = out[0, prompt_len:].tolist()
    trace, answer, truncated = split_think(gen_ids, cfg["thinking"])
    return dict(model=TAG, source=row["source"], item=int(row["item"]), lang=row["lang"], sample=sample,
                prompt_len=prompt_len, gen_ids=gen_ids,
                n_gen=len(gen_ids), n_trace=len(trace), n_answer=len(answer),
                truncated=truncated, hit_cap=len(gen_ids) == cfg["max_new_tokens"],
                rep_trace=distinct_ngram_ratio(trace), rep_answer=distinct_ngram_ratio(answer),
                secs=round(secs, 1), tok_per_s=round(len(gen_ids) / secs, 1),
                trace_text=tok.decode(trace), answer_text=tok.decode(answer))

cache: load saved generations instead of regenerating

In [ ]:
from pathlib import Path

GEN_DIR = Path("../data/generations")
GEN_DIR.mkdir(parents=True, exist_ok=True)

def run_or_load(path, run_fn, rerun=False, allow_run=True):
    """If `path` exists, load its records; else call run_fn() -> list[dict] and save it as JSONL.
    Generations are expensive: pass rerun=True to regenerate on purpose (overwrites the file).
    allow_run=False: if there's no saved file, skip instead of generating (returns [])."""
    if path.exists() and not rerun:
        records = pd.read_json(path, lines=True).to_dict("records")
        print(f"loaded {len(records)} records from {path.name}  (rerun=True to regenerate)")
        return records
    if not allow_run:
        print(f"{path.name} not found and allow_run=False: skipped")
        return []
    records = run_fn()
    pd.DataFrame(records).to_json(path, orient="records", lines=True, force_ascii=False)
    print(f"saved {len(records)} records to {path.name}")
    return records

quality sweep: 1 sample x first 3 items x (source, lang) = 18 generations

In [ ]:
def run_sweep():
    sweep_rows = (pilot.groupby(["source", "lang"], sort=False)
                       .head(3)                      # first 3 items of each (source, lang)
                       .reset_index(drop=True))
    print(len(sweep_rows), "generations")            # 18
    out = []
    for _, row in sweep_rows.iterrows():
        r = generate_one(row)
        out.append(r)
        print(f'{r["source"]:13s} {r["lang"]} item={r["item"]:3d}  n_gen={r["n_gen"]:4d}  '
              f'trunc={r["truncated"]!s:5s}  rep={r["rep_trace"]}/{r["rep_answer"]}  '
              f'{r["secs"]:6.1f}s  {r["tok_per_s"]} tok/s')
    return out

sweep = run_or_load(GEN_DIR / f"b1_sweep_{TAG}.jsonl", run_sweep, allow_run=RUN_SWEEP)
sweep_df = pd.DataFrame(sweep)
if sweep:
    display(sweep_df.groupby(["source", "lang"])[["n_gen", "truncated", "rep_answer", "tok_per_s"]].mean())

read the outputs

In [ ]:
for r in sweep:
    print(f'===== {r["source"]} | {r["lang"]} | item {r["item"]} =====')
    if r["trace_text"]:
        print("TRACE:", r["trace_text"][:300].replace("\n", " "), "...")
    print("ANSWER:", r["answer_text"][:400].replace("\n", " "), "\n")

forcing X: natural vs prefill (3 math items x ne/hi/ru x 2 conditions = 18 generations; 1.7B file also has instr)

In [ ]:
# Instruction appended to the user message, written in X itself (no Latin)
INSTR = {
    "ne": "कृपया आफ्नो सोच्ने प्रक्रिया पनि पूर्ण रूपमा नेपालीमा लेख्नुहोस्।",
    "hi": "कृपया अपनी सोचने की प्रक्रिया भी पूरी तरह हिंदी में लिखें।",
    "ru": "Пожалуйста, веди все свои рассуждения тоже полностью на русском языке.",
}
# First words of the trace, written by us after "<think>\n" (no trailing whitespace: avoids token-merge issues at the boundary)
PREFILL = {
    "ne": "ठीक छ, नेपालीमा सोचौं। पहिले प्रश्नलाई राम्ररी बुझौं।",
    "hi": "ठीक है, चलिए हिंदी में सोचते हैं। पहले प्रश्न को ध्यान से समझते हैं।",
    "ru": "Хорошо, давайте рассуждать на русском. Сначала внимательно разберём условие задачи.",
}

def build_inputs(row, cond):
    """cond in {"natural", "instr", "prefill"}. Returns (enc, prefill_text)."""
    lang = row["lang"]
    content = row["prompt"] + ("\n\n" + INSTR[lang] if cond == "instr" else "")
    text = tok.apply_chat_template([{"role": "user", "content": content}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=True)
    prefill = "<think>\n" + PREFILL[lang] if cond == "prefill" else ""
    enc = tok(text + prefill, return_tensors="pt", add_special_tokens=False)
    return enc, prefill

# sanity: rendering then tokenizing == apply_chat_template's own ids; prefill starts with the real <think> token
r0 = pilot[(pilot.source == "mclm_math100") & (pilot.lang == "ne")].iloc[0]
ref = tok.apply_chat_template([{"role": "user", "content": r0["prompt"]}], add_generation_prompt=True,
                              enable_thinking=True, return_tensors="pt", return_dict=True)["input_ids"]
assert torch.equal(build_inputs(r0, "natural")[0]["input_ids"], ref)
enc_p, _ = build_inputs(r0, "prefill")
assert tok.convert_tokens_to_ids("<think>") in enc_p["input_ids"][0, ref.shape[1]:].tolist()
print(tok.decode(enc_p["input_ids"][0, ref.shape[1] - 3:]))

In [ ]:
@torch.no_grad()
def generate_cond(row, cond, sample=0):
    cfg = SETTINGS[row["source"]]
    enc, prefill = build_inputs(row, cond)
    enc = enc.to(device)
    prompt_len = enc["input_ids"].shape[1]

    torch.manual_seed(seed_for(row["source"], row["item"], row["lang"], sample))
    t0 = time.perf_counter()
    out = model.generate(**enc, do_sample=True,
                         max_new_tokens=cfg["max_new_tokens"],
                         temperature=cfg["temperature"],
                         top_p=cfg["top_p"], top_k=cfg["top_k"])
    if device == "mps":
        torch.mps.synchronize()
    secs = time.perf_counter() - t0

    gen_ids = out[0, prompt_len:].tolist()          # model's own tokens only (prefill is part of the prompt)
    trace, answer, truncated = split_think(gen_ids, cfg["thinking"])
    return dict(model=TAG, source=row["source"], item=int(row["item"]), lang=row["lang"], cond=cond, sample=sample,
                prompt_len=prompt_len, prefill=prefill, gen_ids=gen_ids,
                n_gen=len(gen_ids), n_trace=len(trace), n_answer=len(answer), truncated=truncated,
                rep_trace=distinct_ngram_ratio(trace), secs=round(secs, 1),
                trace_text=tok.decode(trace), answer_text=tok.decode(answer))

rough word-level script labels (test only; the real labeller comes later)

In [ ]:
import regex as re

MATHX = re.compile(r"\$\$.*?\$\$|\$.*?\$|\\\(.*?\\\)|\\\[.*?\\\]", re.S)   # $$..$$, $..$, \(..\), \[..\]
CMD   = re.compile(r"\\[a-zA-Z]+")                                      # leftover LaTeX commands
WORD  = re.compile(r"[\p{L}\p{M}]+")                                     # letters + combining marks (matras!)
SCRIPT = {"ne": "Devanagari", "hi": "Devanagari", "ru": "Cyrillic"}

def word_labels(text, lang):
    """Rough per-word labels: X = prompt-language script, L = Latin, O = other. Skips 1-letter Latin words."""
    text = text.replace("<think>", " ").replace("</think>", " ")   # special tokens, not words ("think" = Latin!)
    text = CMD.sub(" ", MATHX.sub(" ", text))
    words, labels = [], []
    for w in WORD.findall(text):
        if re.fullmatch(rf"[\p{{{SCRIPT[lang]}}}\p{{M}}]+", w):
            lab = "X"
        elif re.fullmatch(r"\p{Latin}+", w):
            lab = "L"
        else:
            lab = "O"
        if lab == "L" and len(w) < 2:        # single Latin letters = math variables; but छ, म, र are real words
            continue
        words.append(w); labels.append(lab)
    return words, labels

def first_run(labels, lab="L", k=5):
    """Index of the first word starting a run of >= k consecutive `lab` words, else None."""
    run = 0
    for i, l in enumerate(labels):
        run = run + 1 if l == lab else 0
        if run == k:
            return i - k + 1
    return None

def trace_stats(r):
    words, labels = word_labels(r["trace_text"], r["lang"])
    nx, nl = labels.count("X"), labels.count("L")
    return dict(n_words=len(words),
                start=labels[0] if labels else None,                 # model's first own word
                x_share=round(nx / (nx + nl), 2) if nx + nl else None,
                drift_word=first_run(labels, "L", 5))                # first sustained (>=5 words) Latin run

run

In [ ]:
FORCE_ITEMS = [5, 9, 31]
CONDS = ["natural", "prefill"]               # "instr" dropped: ignored by 1.7B (0/6 compliance)

def run_force():
    force_rows = pilot[(pilot.source == "mclm_math100") & pilot["item"].isin(FORCE_ITEMS)]
    print(len(force_rows) * len(CONDS), "generations")              # 18
    out = []
    for _, row in force_rows.iterrows():
        for cond in CONDS:
            r = generate_cond(row, cond)
            out.append(r)
            print(f'{r["lang"]} item={r["item"]:3d} {cond:8s} n_gen={r["n_gen"]:4d} trunc={r["truncated"]!s:5s} '
                  f'rep={r["rep_trace"]}  {r["secs"]:5.1f}s  {round(r["n_gen"] / r["secs"], 1)} tok/s')
    return out

force = run_or_load(GEN_DIR / f"b1_force_test_{TAG}.jsonl", run_force)
for r in force:                       # stats are recomputed from the saved text, so labeller fixes apply to cached runs
    r.update(trace_stats(r))
force_df = pd.DataFrame(force)

summary and drift points

In [ ]:
force_df["start_X"] = force_df["start"].eq("X")
force_df["drifted"] = force_df["drift_word"].notna()
force_df.groupby(["lang", "cond"], sort=False).agg(
    start_X=("start_X", "mean"), x_share=("x_share", "mean"),
    drifted=("drifted", "mean"), drift_word=("drift_word", "median"),
    n_words=("n_words", "median"), truncated=("truncated", "mean"))

In [ ]:
def show_drift(r, before=15, after=15):
    words, labels = word_labels(r["trace_text"], r["lang"])
    d = r["drift_word"]
    if pd.isna(d):                                   # None when fresh, NaN after loading from JSONL
        return "(no sustained Latin run)"
    d = int(d)
    return " ".join(words[max(0, d - before):d]) + "  ▶▶  " + " ".join(words[d:d + after])

for r in force:
    print(f'===== {r["lang"]} | item {r["item"]} | {r["cond"]} | start={r["start"]} x_share={r["x_share"]} =====')
    print("TRACE START:", r["trace_text"][:].replace("\n", " "))
    print("AT DRIFT:   ", show_drift(r), "\n")

compare models: every saved force-test file (1.7B vs 4B)

In [ ]:
frames = []
for f in sorted(GEN_DIR.glob("b1_force_test_*.jsonl")):
    d = pd.read_json(f, lines=True)
    d["model"] = f.stem.removeprefix("b1_force_test_")          # filename is the source of truth for old files
    st = pd.DataFrame([trace_stats(r) for r in d.to_dict("records")])
    frames.append(pd.concat([d.drop(columns=st.columns, errors="ignore"), st], axis=1))
allf = pd.concat(frames, ignore_index=True)
allf["start_X"] = allf["start"].eq("X")
allf["drifted"] = allf["drift_word"].notna()
(allf[allf.cond != "instr"]
     .groupby(["lang", "cond", "model"], sort=False)
     .agg(start_X=("start_X", "mean"), x_share=("x_share", "mean"), drifted=("drifted", "mean"),
          drift_word=("drift_word", "median"), rep=("rep_trace", "mean"), truncated=("truncated", "mean"))
     .round(2))

In [ ]:
# C2-0 / H11: same math items with thinking OFF. Does the thinking phase (not the question) pull ne/hi into English?
OFF_CFG = dict(max_new_tokens=2048, temperature=0.7, top_p=0.8, top_k=20)   # Qwen3's recommended non-thinking settings

@torch.no_grad()
def generate_think_off(row, sample=0):
    enc = tok.apply_chat_template([{"role": "user", "content": row["prompt"]}], add_generation_prompt=True,
                                  enable_thinking=False, return_tensors="pt", return_dict=True).to(device)
    prompt_len = enc["input_ids"].shape[1]
    torch.manual_seed(seed_for(row["source"], row["item"], row["lang"], sample))
    t0 = time.perf_counter()
    out = model.generate(**enc, do_sample=True, **OFF_CFG)
    if device == "mps":
        torch.mps.synchronize()
    secs = time.perf_counter() - t0
    gen_ids = out[0, prompt_len:].tolist()
    answer = [t for t in gen_ids if t not in EOS_IDS]
    return dict(model=TAG, source=row["source"], item=int(row["item"]), lang=row["lang"], cond="think_off",
                sample=sample, prompt_len=prompt_len, gen_ids=gen_ids, n_gen=len(gen_ids),
                hit_cap=len(gen_ids) == OFF_CFG["max_new_tokens"], rep_answer=distinct_ngram_ratio(answer),
                secs=round(secs, 1), answer_text=tok.decode(answer))

# sanity: with thinking off, the template itself pre-fills an empty think block at the end of the prompt
r0 = pilot[(pilot.source == "mclm_math100") & (pilot.lang == "ne")].iloc[0]
ids0 = tok.apply_chat_template([{"role": "user", "content": r0["prompt"]}], add_generation_prompt=True,
                               enable_thinking=False, return_tensors="pt", return_dict=True)["input_ids"][0]
print(repr(tok.decode(ids0[-8:])))
assert THINK_END in ids0.tolist()

def run_think_off():
    rows = pilot[(pilot.source == "mclm_math100") & pilot["item"].isin(FORCE_ITEMS)]
    out = []
    for _, row in rows.iterrows():
        r = generate_think_off(row)
        out.append(r)
        print(f'{r["lang"]} item={r["item"]:3d} n_gen={r["n_gen"]:4d} cap={r["hit_cap"]!s:5s} '
              f'rep={r["rep_answer"]}  {r["secs"]:5.1f}s')
    return out

off = run_or_load(GEN_DIR / f"b1_think_off_{TAG}.jsonl", run_think_off)

def text_stats(text, lang):
    words, labels = word_labels(text, lang)
    nx, nl = labels.count("X"), labels.count("L")
    return dict(n_words=len(words), start=labels[0] if labels else None,
                x_share=round(nx / (nx + nl), 2) if nx + nl else None)

rows = []
for r in force:
    if r["cond"] == "natural":
        rows.append(dict(lang=r["lang"], item=r["item"], part="on: trace",  **text_stats(r["trace_text"], r["lang"])))
        rows.append(dict(lang=r["lang"], item=r["item"], part="on: answer", **text_stats(r["answer_text"], r["lang"])))
for r in off:
    rows.append(dict(lang=r["lang"], item=r["item"], part="off: answer", **text_stats(r["answer_text"], r["lang"])))
cmp = pd.DataFrame(rows)
cmp["start_X"] = cmp["start"].eq("X")
display(cmp.groupby(["lang", "part"], sort=False)[["start_X", "x_share", "n_words"]].mean().round(2))

for r in off:
    print(f'===== {r["lang"]} | item {r["item"]} | think_off =====')
    print(r["answer_text"][:300].replace("\n", " "), "\n")

In [ ]:
import textwrap

PROMPTS = pilot.set_index(["source", "item", "lang"])["prompt"]

def clip(text, head=800, tail=400):
    """Long traces: show the start and the end, skip the middle."""
    if len(text) <= head + tail:
        return text
    return f"{text[:head]}\n\n   [... {len(text) - head - tail} chars skipped ...]\n\n{text[-tail:]}"

def show_full(item, lang, cond="natural", head=800, tail=400):
    """Prompt -> thinking trace -> final answer for one generation. cond: 'natural', 'prefill', or 'think_off'."""
    pool = off if cond == "think_off" else [r for r in force if r["cond"] == cond]
    r = next(r for r in pool if r["item"] == item and r["lang"] == lang)
    bar = "=" * 100
    print(f"{bar}\n{lang} | item {item} | {cond} | n_gen={r['n_gen']} | truncated={r.get('truncated', False)}\n{bar}")
    print("\n--- PROMPT ---\n" + PROMPTS[(r["source"], item, lang)])
    if cond != "think_off":
        trace = r.get("prefill", "") + r["trace_text"]      # prefill = the words we forced after <think>
        print(f"\n--- THINKING ({len(trace)} chars) ---\n" + clip(trace, head, tail))
    print(f"\n--- ANSWER ({len(r['answer_text'])} chars) ---\n" + clip(r["answer_text"], head, tail))

# the interesting contrast: same item, English trace -> answer back in X (item 9) vs answer stays English (item 5)
show_full(9, "ne")
show_full(9, "ne", "prefill")
show_full(9, "ne", "think_off")
show_full(5, "ne")
show_full(5, "ne", "prefill")
show_full(5, "ne", "think_off")
